# Messages (LangChain 1.3+)

Messages are the **fundamental unit of context** for chat models. They carry everything the model needs to represent a conversation:

| Field | Purpose |
|-------|---------|
| **Role** | Message type (`system`, `user`/`human`, `assistant`/`ai`, `tool`) |
| **Content** | Text, images, audio, files (multimodal where supported) |
| **Metadata** | Token usage, message IDs, `tool_calls`, provider fields |

LangChain provides standard message classes that work across OpenAI, Google, Groq, etc.

**Prerequisite:** `GROQ_API_KEY` in repo-root `.env`. Prior notebooks: [2-modelintegration](2-modelintegration.ipynb) (models), [3_tools](3_tools.ipynb) (tool calls).

**Sections:** 1 text prompts | 2 message types + examples | 3 full trail | 4 dict shorthand

In [3]:
"""Load API keys from .env before any model calls."""
import os
from pathlib import Path

from dotenv import load_dotenv

for env_path in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if env_path.is_file():
        load_dotenv(env_path)
        break
else:
    load_dotenv()

for key in ("GROQ_API_KEY",):
    value = os.getenv(key)
    if value:
        os.environ[key] = value
    else:
        print(f"{key}: missing (add to .env)")

In [4]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")
model  # ChatGroq instance

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x112936720>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x112a067e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

## 1. Text prompts (string)

Pass a **plain string** to `model.invoke()`. LangChain wraps it as a single `HumanMessage` internally.

**Use text prompts when:**

- You have one standalone question
- You do not need conversation history
- You want minimal code

**Returns:** an `AIMessage` (the model's reply).

In [5]:
# str in → HumanMessage (implicit) → API → AIMessage out
response = model.invoke("Who are you?")

print(type(response).__name__)  # AIMessage
print(response.content[:200], "...")  # first 200 chars of reply

AIMessage
<think>
Okay, the user is asking who I am. I should guide the user to check the official website, public documentation, or technical reports for detailed information. I need to be clear and polite, av ...


## 2. Message prompts (list of messages)

Pass a **list of message objects** when you need system instructions, chat history, or tool results.

### Message types

| Class | Role | Purpose |
|-------|------|---------|
| **`SystemMessage`** | `system` | Primes behavior — tone, role, guidelines (not user-facing) |
| **`HumanMessage`** | `user` / `human` | User input — text, images, audio, files |
| **`AIMessage`** | `assistant` / `ai` | Model output — text, `tool_calls`, provider metadata |
| **`ToolMessage`** | `tool` | Result of one tool execution, linked by `tool_call_id` |

Import from `langchain_core.messages`.

### 2a. `SystemMessage`

An initial set of instructions that **primes** the model: set tone, define role, set response rules. Usually placed **first** in the list.

In [6]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a concise travel assistant. Answer in one short sentence."),
    HumanMessage(content="What is the capital of Japan?"),
]

response = model.invoke(messages)
response.content

"<think>\nOkay, the user is asking for the capital of Japan. I need to confirm the correct answer. While Tokyo is widely recognized as the capital, some might confuse it with Kyoto, which was the capital in the past. However, since the 19th century, the capital has been Tokyo. I should state clearly that Tokyo is the capital. Keep the answer short and direct as per the user's request. No need for extra details. Just one sentence.\n</think>\n\nThe capital of Japan is Tokyo."

### 2b. `HumanMessage`

Represents **user input**. For multi-turn chat, append each new user turn as another `HumanMessage` (along with prior `AIMessage`s).

In [7]:
from langchain_core.messages import AIMessage, HumanMessage

# Two-turn conversation: user → assistant → user again
history = [
    HumanMessage(content="My name is Alex."),
    AIMessage(content="Nice to meet you, Alex!"),
    HumanMessage(content="What is my name?"),
]

response = model.invoke(history)
response.content  # model should recall "Alex" from history

'<think>\nOkay, the user asked, "What is my name?" after introducing themselves as Alex. Let me check the history.\n\nIn the previous message, the user clearly said, "My name is Alex." So I need to confirm that I remember that correctly. The user might be testing if I can retain information from the conversation or just wants to ensure I\'m paying attention.\n\nI should respond by stating their name again to confirm. Maybe add a friendly greeting to keep the interaction positive. Let me make sure there\'s no confusion with other names mentioned. Since there\'s no other name in the history, it\'s safe to just say "Alex" again. Also, keeping the response concise but warm would be good. Alright, the answer should be straightforward.\n</think>\n\nYour name is Alex. Nice to meet you again! 😊'

### 2c. `AIMessage`

Represents **model output**. Every `model.invoke(...)` returns an `AIMessage`.

- `.content` — reply text
- `.tool_calls` — list of requested tool invocations (empty if none)
- `.response_metadata` — token usage, `finish_reason`, model name
- Can include multimodal data on supported providers

In [8]:
ai_msg = model.invoke("Say hello in exactly three words.")

print(f"Type:      {type(ai_msg).__name__}")
print(f"Content:   {ai_msg.content}")
print(f"Tool calls: {ai_msg.tool_calls}")
print(f"Finish:    {ai_msg.response_metadata.get('finish_reason')}")

Type:      AIMessage
Content:   <think>
Okay, the user wants me to say hello in exactly three words. Let me think... The standard greeting is "Hello" which is one word, but I need three. Maybe I can expand it. How about adding a name or a title? Like "Hello there friend" – that's three words. Wait, but maybe they want something more formal or creative. Let me check if the user has any specific context. The query is pretty straightforward. Maybe "Greetings and salutations" but that's four. Hmm. How about "Hello how are you"? That's three, but it's a question. The user didn't specify if it needs to be a statement or question. Alternatively, "Hi there" is two words. Maybe "Hey hey hey" but that's repetitive. Wait, maybe the user just wants a greeting in three words regardless of structure. Let me think again. "Good day sir" – three words. Or "Salutations to you". Alternatively, "Hi, I'm here" but that's four. Wait, perhaps "Hello world" – but that's a common phrase. Or "Hi there everyone"

### 2d. `ToolMessage`

When an `AIMessage` contains **`tool_calls`**, you run the tool in Python and send the result back as a `ToolMessage`.

- `.content` — observation string (what the tool returned)
- `.tool_call_id` — must match the `id` from the `AIMessage.tool_calls` entry

See **[3_tools.ipynb](3_tools.ipynb)** for the full ReAct loop.

In [9]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"It's sunny and 72°F in {city}."


model_with_tools = model.bind_tools([get_weather])

# Step 1: model requests a tool call (AIMessage with tool_calls)
ai_with_tools = model_with_tools.invoke([HumanMessage(content="Weather in Paris?")])
print("Tool calls:", ai_with_tools.tool_calls)

# Step 2: run tool → ToolMessage
tool_call = ai_with_tools.tool_calls[0]
tool_msg = ToolMessage(
    content=get_weather.invoke(tool_call["args"]),
    tool_call_id=tool_call["id"],
)
print(f"ToolMessage: {tool_msg.content}")

# Step 3: model reads tool result → final AIMessage
final = model_with_tools.invoke([HumanMessage(content="Weather in Paris?"), ai_with_tools, tool_msg])
final.content

Tool calls: [{'name': 'get_weather', 'args': {'city': 'Paris'}, 'id': '1zaj7gx3a', 'type': 'tool_call'}]
ToolMessage: It's sunny and 72°F in Paris.


'The current weather in Paris is sunny with a temperature of 72°F. 🌞 Let me know if you need more details!'

## 3. Full message trail (all four types)

Typical agent turn with tools:

```
SystemMessage → HumanMessage → AIMessage (tool_calls) → ToolMessage → AIMessage (answer)
```

| # | Type | Who |
|---|------|-----|
| 0 | `SystemMessage` | Your instructions |
| 1 | `HumanMessage` | User question |
| 2 | `AIMessage` | Model chooses tool |
| 3 | `ToolMessage` | Your code runs tool |
| 4 | `AIMessage` | Model final answer |

In [10]:
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)

messages = [SystemMessage(content="You are a helpful weather bot.")]
messages.append(HumanMessage(content="What's the weather in London?"))

ai_step = model_with_tools.invoke(messages)
messages.append(ai_step)

for tc in ai_step.tool_calls:
    messages.append(
        ToolMessage(content=get_weather.invoke(tc["args"]), tool_call_id=tc["id"])
    )

final_ai = model_with_tools.invoke(messages)
messages.append(final_ai)

for i, msg in enumerate(messages):
    print(f"{i}. {type(msg).__name__}: {str(msg.content)[:60]}...")

0. SystemMessage: You are a helpful weather bot....
1. HumanMessage: What's the weather in London?...
2. AIMessage: ...
3. ToolMessage: It's sunny and 72°F in London....
4. AIMessage: The current weather in London is sunny with a temperature of...


## 4. Dict shorthand (optional)

LangChain also accepts `{"role": "...", "content": "..."}` dicts. Roles map to message classes automatically:

| Dict `role` | Message class |
|-------------|---------------|
| `"system"` | `SystemMessage` |
| `"user"` / `"human"` | `HumanMessage` |
| `"assistant"` / `"ai"` | `AIMessage` |
| `"tool"` | `ToolMessage` (needs `tool_call_id`) |

In [11]:
# Same as SystemMessage + HumanMessage, using dicts
response = model.invoke([
    {"role": "system", "content": "Reply in exactly one word."},
    {"role": "user", "content": "Color of the sky on a clear day?"},
])
response.content

'<think>\nOkay, the user is asking for the color of the sky on a clear day. Let me think about this. I know that the sky appears blue when the sun is up and there are no clouds. But why exactly is it blue? Oh right, it\'s because of Rayleigh scattering. The Earth\'s atmosphere scatters sunlight in all directions, and blue light is scattered more than other colors because it travels in shorter, smaller waves. So even though the sun emits all colors, the blue gets scattered more, making the sky look blue from the ground. Wait, but sometimes it can look a bit different, like a pale blue or even a bit white near the sun. But under clear conditions, the standard answer is blue. The user wants exactly one word, so "blue" should be the correct answer. Let me double-check if there are any exceptions. In places with a lot of pollution, maybe it\'s more white or hazy, but the question specifies a clear day. So yeah, blue is the right answer.\n</think>\n\nblue'

### Summary

| Input to `invoke` | When |
|-------------------|------|
| `str` | Single question, no history |
| `[SystemMessage, HumanMessage, ...]` | Instructions + history + tools |
| `[{"role": "user", ...}, ...]` | Same as above, dict style |

```python
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

model.invoke("hello")                              # str → AIMessage
model.invoke([HumanMessage(content="hello")])      # explicit messages
model.invoke([{"role": "user", "content": "hi"}])  # dict shorthand
```

**Next:** Use messages inside `create_agent` — see [1_lang_chain_intro.ipynb](1_lang_chain_intro.ipynb).